# Module 11: Distributed Caching Stampede Prevention — Interactive Laboratory

Every cell below runs the module's **real** implementation from
`project_solution/distributed_cache_guard.py`. Nothing here prints a claim it has not verified.

What you will do:

1. Load the engine and inspect what it actually exports.
2. Run its primary workflow and check the assertions that define correctness.
3. **Commit to a prediction**, then run the cell that tests it.
4. Measure a property rather than asserting one.
5. Fix a deliberately broken cell in place.

> The code in cells 4, 6 and 8 is lifted from this module's own test suite, so it
> cannot drift from the implementation. If the API changes, those tests fail
> first and this notebook is regenerated from them.


## 1. Load the engine and introspect it

Rather than trusting a hardcoded list of class names, ask the module what it
actually contains.


In [ ]:
import inspect
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve() / 'project_solution'))
import distributed_cache_guard

classes = [n for n, o in inspect.getmembers(distributed_cache_guard, inspect.isclass)
           if o.__module__ == 'distributed_cache_guard']
functions = [n for n, o in inspect.getmembers(distributed_cache_guard, inspect.isfunction)
             if o.__module__ == 'distributed_cache_guard']

print('module   : distributed_cache_guard')
print(f'classes  : {classes}')
print(f'functions: {functions}')
print()
for name in classes:
    obj = getattr(distributed_cache_guard, name)
    try:
        sig = inspect.signature(obj.__init__)
        params = [p for p in sig.parameters if p != 'self']
    except (TypeError, ValueError):
        params = ['<builtin>']
    print(f'  {name}({", ".join(params)})')

## 2. Baseline: Cache hit and miss lifecycle

This is the module's own `test_cache_hit_and_miss_lifecycle` — real instantiation, real calls, real
assertions. If it runs clean, the property it encodes holds.


In [ ]:
import concurrent.futures
import time

from distributed_cache_guard import (
    DistributedCacheGuard,
)

cache = DistributedCacheGuard()
db_reads = {"count": 0}

def db_loader() -> str:
    db_reads["count"] += 1
    return "expensive_user_profile_data"

# 1. First get: Cache Miss -> calls loader
val1 = cache.get("user:101", db_loader, ttl_seconds=60.0)
assert val1 == "expensive_user_profile_data"
assert db_reads["count"] == 1
assert cache.stats["hits"] == 0
assert cache.stats["misses"] == 1

# 2. Second get: Cache Hit -> does NOT call loader
val2 = cache.get("user:101", db_loader, ttl_seconds=60.0)
assert val2 == "expensive_user_profile_data"
assert db_reads["count"] == 1
assert cache.stats["hits"] == 1

print('PASSED: test_cache_hit_and_miss_lifecycle')

## 3. 🔮 Prediction — commit before you run

50 concurrent requests arrive for a key that just expired. Predict how many reach the database. Write the number down before running the next cell.

Write your answer down. An uncommitted guess teaches nothing, because you will
retro-fit it to whatever the next cell prints.

The next cell runs `test_single_flight_collapses_stampede`, which tests exactly this property.


In [ ]:
cache = DistributedCacheGuard()
db_reads = {"count": 0}

def slow_db_loader() -> dict[str, int]:
    time.sleep(0.05)  # simulate 50ms database latency
    db_reads["count"] += 1
    return {"sku_1": 100}

# Fire 50 threads simultaneously for the same key
with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
    futures = [executor.submit(cache.get, "hot_item_inventory", slow_db_loader) for _ in range(50)]
    results = [f.result() for f in concurrent.futures.as_completed(futures)]

# All 50 threads received the correct data
assert len(results) == 50
for r in results:
    assert r == {"sku_1": 100}

# Despite 50 concurrent requests, the database was hit EXACTLY ONCE!
assert db_reads["count"] == 1
assert cache.stats["db_loads"] == 1

print('PASSED: test_single_flight_collapses_stampede')

## 4. Measure it: Negative caching protects penetration

An assertion tells you a property holds. A measurement tells you *how much*.
This cell runs `test_negative_caching_protects_penetration` and times it.


In [ ]:

_t0 = time.perf_counter()

cache = DistributedCacheGuard()
db_reads = {"count": 0}

def missing_db_loader() -> None:
    db_reads["count"] += 1
    return None  # record does not exist in DB

# Request missing user: DB checked once
res1 = cache.get("non_existent_user_999", missing_db_loader, ttl_seconds=60.0, negative_ttl=2.0)
assert res1 is None
assert db_reads["count"] == 1

# Second request within negative_ttl returns None from cache without querying DB
res2 = cache.get("non_existent_user_999", missing_db_loader, ttl_seconds=60.0, negative_ttl=2.0)
assert res2 is None
assert db_reads["count"] == 1
assert cache.stats["hits"] == 1

_elapsed = (time.perf_counter() - _t0) * 1000
print('PASSED: test_negative_caching_protects_penetration')
print(f'wall clock: {_elapsed:.2f} ms')

## 5. 🛠️ Fix this cell — it is deliberately broken

The cell below asserts something **false** about the real object. Read the
failure, work out the true value from the module's actual behaviour, and correct
the expected number.

Do not delete the assertion. The point is to make it pass by knowing the answer.


In [ ]:
# DELIBERATELY BROKEN - fix the expected value below.
# Hint: print the real value first, then decide what the assertion should say.

exports = [n for n in dir(distributed_cache_guard) if not n.startswith('_')]
print(f'actual export count: {len(exports)}')
print(f'actual exports     : {exports}')

EXPECTED_EXPORT_COUNT = 999      # <-- wrong on purpose. Replace it.

assert len(exports) == EXPECTED_EXPORT_COUNT, (
    f'expected {EXPECTED_EXPORT_COUNT} exports, found {len(exports)}. '
    'Read the printed value above and correct the constant.'
)
print('Fixed - assertion now reflects reality.')

### 🎓 Key takeaways

1. Single-flight turns N concurrent misses into one load. Measure it, do not assume it.
2. Probabilistic early expiry removes the synchronised expiry cliff entirely.
3. Negative caching is what stops a nonexistent key from becoming a DDoS.

---

**Continue with this module:**

- [README.md](README.md) — the mental model and failure modes
- [PROJECT_GUIDE.md](PROJECT_GUIDE.md) — build it yourself, in 3 tiers
- [starter/](starter/) — your stubs; run the tests from there to grade yourself
- [debug_lab/SYMPTOMS.md](debug_lab/SYMPTOMS.md) — diagnose planted bugs from the symptom
- [TROUBLESHOOTING_AND_EDGE_CASES.md](TROUBLESHOOTING_AND_EDGE_CASES.md) — real errors, real causes
- [SELF_ASSESSMENT_AND_CHALLENGES.md](SELF_ASSESSMENT_AND_CHALLENGES.md) — quiz and diagnostics
